In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

UNIFIED_FONT_FAMILY = 'DejaVu Sans'
# Adjust these font sizes manually to tune the exported figures.
UNIFIED_FONT_SIZE = 13
AXIS_LABEL_FONT_SIZE = 13
TITLE_FONT_SIZE = 13
TICK_FONT_SIZE = 12
ANNOTATION_FONT_SIZE = 9
LEGEND_FONT_SIZE = 12
MARKER_AREA_MIN = 120
MARKER_AREA_MAX = 220
PANEL_WSPACE = 0.34
FIGURE_BOTTOM_MARGIN = 0.22
LEGEND_VERTICAL_OFFSET = -0.06
FIGURE_FACE = 'white'
AXES_FACE = 'white'
AXIS_TEXT_COLOR = '#4A4A4A'
FRAME_COLOR = '#4A4A4A'
GREY = '#878787'
MATTE_BLUE = '#2F4F6F'
LIGHT_BLUE = '#BDD9E8'
PALE_BLUE = '#DBEEF5'
EDGE_COLOR = '#173042'
GRID_COLOR = '#edf2f7'
BACKGROUND = AXES_FACE
LEGEND_FACE = 'white'
LEGEND_EDGE = FRAME_COLOR
ERRORBAR_COLOR = '#6f6f6f'
BAR_MAC_COLOR = '#6F8FB8'
BAR_AC_COLOR = '#9FC7B9'
HIGHLIGHT_RED = '#d62728'

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = UNIFIED_FONT_FAMILY
mpl.rcParams['font.size'] = UNIFIED_FONT_SIZE
mpl.rcParams['axes.labelsize'] = AXIS_LABEL_FONT_SIZE
mpl.rcParams['axes.titlesize'] = TITLE_FONT_SIZE
mpl.rcParams['xtick.labelsize'] = TICK_FONT_SIZE
mpl.rcParams['ytick.labelsize'] = TICK_FONT_SIZE
mpl.rcParams['legend.fontsize'] = LEGEND_FONT_SIZE
mpl.rcParams['axes.labelcolor'] = AXIS_TEXT_COLOR
mpl.rcParams['axes.edgecolor'] = FRAME_COLOR
mpl.rcParams['xtick.color'] = AXIS_TEXT_COLOR
mpl.rcParams['ytick.color'] = AXIS_TEXT_COLOR

MODEL_COLORS = {
    'LSTM': '#bdd9e8',
    'MLP': '#dbeef5',
    'SOTA GRU': GREY,
    'Ours': MATTE_BLUE,
}


def style_single_axis(ax, xlabel=None, ylabel=None):
    ax.set_facecolor(AXES_FACE)
    ax.set_axisbelow(True)
    for spine_name in ['left', 'right', 'top', 'bottom']:
        ax.spines[spine_name].set_visible(True)
        ax.spines[spine_name].set_color(FRAME_COLOR)
        ax.spines[spine_name].set_linewidth(1.0)
    ax.tick_params(axis='x', colors=AXIS_TEXT_COLOR, length=0)
    ax.tick_params(axis='y', colors=AXIS_TEXT_COLOR)
    if xlabel is not None:
        ax.set_xlabel(xlabel, color=AXIS_TEXT_COLOR, fontweight='bold')
    if ylabel is not None:
        ax.set_ylabel(ylabel, color=AXIS_TEXT_COLOR, fontweight='bold')


def style_dual_axis(
    ax_left,
    ax_right,
    xlabel=None,
    ylabel_left=None,
    ylabel_right=None,
    left_color=AXIS_TEXT_COLOR,
    right_color=MATTE_BLUE,
):
    style_single_axis(ax_left, xlabel=xlabel, ylabel=None)
    ax_left.tick_params(axis='y', colors=left_color)
    ax_left.spines['left'].set_color(left_color)
    if ylabel_left is not None:
        ax_left.set_ylabel(ylabel_left, color=left_color, fontweight='bold')

    ax_right.patch.set_alpha(0)
    ax_right.tick_params(axis='y', colors=right_color)
    ax_right.spines['right'].set_visible(True)
    ax_right.spines['right'].set_color(right_color)
    ax_right.spines['right'].set_linewidth(1.0)
    ax_right.spines['top'].set_color(FRAME_COLOR)
    ax_right.spines['top'].set_linewidth(1.0)
    if ylabel_right is not None:
        ax_right.set_ylabel(ylabel_right, color=right_color, fontweight='bold')


def style_legend(legend):
    legend.get_frame().set_facecolor(LEGEND_FACE)
    legend.get_frame().set_edgecolor(LEGEND_EDGE)
    legend.get_frame().set_linewidth(1.0)
    legend.get_title().set_color(AXIS_TEXT_COLOR)
    legend.get_title().set_fontsize(LEGEND_FONT_SIZE)
    legend.get_title().set_ha('center')
    legend._legend_box.align = 'center'
    for text in legend.get_texts():
        text.set_color(AXIS_TEXT_COLOR)
        text.set_fontsize(LEGEND_FONT_SIZE)
        text.set_ha('center')


In [4]:
INPUT_FEATURES = 512
OUTPUT_SIZE = 41
BIN_SIZE_MS = 20
BITS_PER_PARAM = 32 # assuming 32-bit floating point parameters
ENERGY_MAC = 4.6e-12 # energy per MAC operation in joules
ENERGY_AC = 0.9e-12 # energy per AC operation in joules
PATCH_STRIDE = 4
RAW_LENGTHS_CSV = "raw_lengths_all_splits.csv"  # change path

import pandas as pd
import numpy as np

## MLP

In [5]:
def raw_to_effective_length(raw_length, patch_size, patch_stride):
    raw_length = int(raw_length)
    if raw_length < patch_size:
        return 0
    return ((raw_length - patch_size) // patch_stride) + 1

def compute_mlp_metrics_from_csv(
    layer_sizes,
    raw_lengths_csv,
    patch_size,
    patch_stride,
):
    df = pd.read_csv(raw_lengths_csv)

    params = 0
    macs_per_step = 0
    acs_per_step = 0

    for i in range(len(layer_sizes) - 1):
        n_in = layer_sizes[i]
        n_out = layer_sizes[i + 1]

        weights = n_in * n_out
        biases = n_out

        params += weights + biases
        macs_per_step += weights
        acs_per_step += biases

    df["effective_length"] = df["raw_length"].apply(
        lambda x: raw_to_effective_length(x, patch_size, patch_stride)
    )

    energy_per_step_j = macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC

    df["macs"] = df["effective_length"] * macs_per_step
    df["acs"] = df["effective_length"] * acs_per_step
    df["energy_J"] = df["effective_length"] * energy_per_step_j
    df["energy_mJ"] = df["energy_J"] * 1e3
    df["energy_uJ"] = df["energy_J"] * 1e6

    memory_bits = params * BITS_PER_PARAM
    memory_bytes = memory_bits / 8
    memory_kb = memory_bytes / 1024
    memory_mb = memory_bytes / 1024**2

    binning_latency_ms = (patch_size - 1) * BIN_SIZE_MS

    aggregate = {
        "mean_effective_length": df["effective_length"].mean(),
        "median_effective_length": df["effective_length"].median(),
        "p95_effective_length": df["effective_length"].quantile(0.95),

        "mean_macs": df["macs"].mean(),
        "median_macs": df["macs"].median(),
        "p95_macs": df["macs"].quantile(0.95),

        "mean_acs": df["acs"].mean(),
        "median_acs": df["acs"].median(),
        "p95_acs": df["acs"].quantile(0.95),

        "mean_energy_J": df["energy_J"].mean(),
        "median_energy_J": df["energy_J"].median(),
        "p95_energy_J": df["energy_J"].quantile(0.95),

        "mean_energy_uJ": df["energy_uJ"].mean(),
        "median_energy_uJ": df["energy_uJ"].median(),
        "p95_energy_uJ": df["energy_uJ"].quantile(0.95),
    }

    return {
        "layer_sizes": layer_sizes,
        "n_params": params,
        "memory_bits": memory_bits,
        "memory_bytes": memory_bytes,
        "memory_kB": memory_kb,
        "memory_MB": memory_mb,

        "macs_per_step": macs_per_step,
        "acs_per_step": acs_per_step,
        "energy_per_step_J": energy_per_step_j,
        "energy_per_step_uJ": energy_per_step_j * 1e6,

        "patch_size": patch_size,
        "patch_stride": patch_stride,
        "binning_latency_ms": binning_latency_ms,

        "per_utterance_df": df,
        "aggregate": aggregate,
    }


In [6]:
# ===== MLP CONFIG 0.5M =====

patch_size = 8
hidden_size = 128
n_layers = 4

input_dim = INPUT_FEATURES * patch_size
layer_sizes = [input_dim] + [hidden_size] * n_layers + [OUTPUT_SIZE]

mlp_metrics = compute_mlp_metrics_from_csv(
    layer_sizes=layer_sizes,
    raw_lengths_csv=RAW_LENGTHS_CSV,
    patch_size=patch_size,
    patch_stride=PATCH_STRIDE,
)

agg = mlp_metrics["aggregate"]

print("===== MLP 0.5M =====")
print("Layer sizes:", mlp_metrics["layer_sizes"])

print("\n--- Model size ---")
print("Parameters:", mlp_metrics["n_params"])
print("Parameter count matches Optuna reported count: 579,241")
print("Memory footprint (bits):", mlp_metrics["memory_bits"])
print("Memory footprint (MB):", mlp_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per step:", mlp_metrics["macs_per_step"])
print("ACs per step:", mlp_metrics["acs_per_step"])
print("Energy per step (uJ):", mlp_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", mlp_metrics["binning_latency_ms"])

===== MLP 0.5M =====
Layer sizes: [4096, 128, 128, 128, 128, 41]

--- Model size ---
Parameters: 579241
Parameter count matches Optuna reported count: 579,241
Memory footprint (bits): 18535712
Memory footprint (MB): 2.2096290588378906

--- Per-step operations ---
MACs per step: 578688
ACs per step: 553
Energy per step (uJ): 2.6624624999999997

--- Effective sequence length ---
Mean effective length: 219.107180459044
Median effective length: 210.0
P95 effective length: 358.0

--- Operations per inference ---
Mean MACs: 126794696.04548326
Median MACs: 121524480.0
P95 MACs: 207170304.0
Mean ACs: 121166.27079385133
Median ACs: 116130.0
P95 ACs: 197974.0

--- Energy per inference ---
Mean energy (uJ): 583.3646514529374
Median energy (uJ): 559.117125
P95 energy (uJ): 953.161575

--- Binning latency ---
Binning latency (ms): 140


In [7]:
# ===== MLP CONFIG 2M =====

patch_size = 8
hidden_size = 384
n_layers = 5

input_dim = INPUT_FEATURES * patch_size
layer_sizes = [input_dim] + [hidden_size] * n_layers + [OUTPUT_SIZE]

mlp_metrics = compute_mlp_metrics_from_csv(
    layer_sizes=layer_sizes,
    raw_lengths_csv=RAW_LENGTHS_CSV,
    patch_size=patch_size,
    patch_stride=PATCH_STRIDE,
)

agg = mlp_metrics["aggregate"]

print("===== MLP 5M =====")
print("Layer sizes:", mlp_metrics["layer_sizes"])

print("\n--- Model size ---")
print("Parameters:", mlp_metrics["n_params"])
print("Parameter count matches Optuna reported count: 2180393")
print("Memory footprint (bits):", mlp_metrics["memory_bits"])
print("Memory footprint (MB):", mlp_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per step:", mlp_metrics["macs_per_step"])
print("ACs per step:", mlp_metrics["acs_per_step"])
print("Energy per step (uJ):", mlp_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", mlp_metrics["binning_latency_ms"])

===== MLP 5M =====
Layer sizes: [4096, 384, 384, 384, 384, 384, 41]

--- Model size ---
Parameters: 2180393
Parameter count matches Optuna reported count: 2180393
Memory footprint (bits): 69772576
Memory footprint (MB): 8.31753921508789

--- Per-step operations ---
MACs per step: 2178432
ACs per step: 1961
Energy per step (uJ): 10.0225521

--- Effective sequence length ---
Mean effective length: 219.107180459044
Median effective length: 210.0
P95 effective length: 358.0

--- Operations per inference ---
Mean MACs: 477310093.34175617
Median MACs: 457470720.0
P95 MACs: 779878656.0
Mean ACs: 429669.1808801853
Median ACs: 411810.0
P95 ACs: 702038.0

--- Energy per inference ---
Mean energy (uJ): 2196.01313163487
Median energy (uJ): 2104.735941
P95 energy (uJ): 3588.0736518

--- Binning latency ---
Binning latency (ms): 140


In [8]:
# ===== MLP CONFIG 5M =====

patch_size = 10
hidden_size = 640
n_layers = 5

input_dim = INPUT_FEATURES * patch_size
layer_sizes = [input_dim] + [hidden_size] * n_layers + [OUTPUT_SIZE]

mlp_metrics = compute_mlp_metrics_from_csv(
    layer_sizes=layer_sizes,
    raw_lengths_csv=RAW_LENGTHS_CSV,
    patch_size=patch_size,
    patch_stride=PATCH_STRIDE,
)

agg = mlp_metrics["aggregate"]

print("===== MLP 5M =====")
print("Layer sizes:", mlp_metrics["layer_sizes"])

print("\n--- Model size ---")
print("Parameters:", mlp_metrics["n_params"])
print("Parameter count matches Optuna reported count: 4944681")
print("Memory footprint (bits):", mlp_metrics["memory_bits"])
print("Memory footprint (MB):", mlp_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per step:", mlp_metrics["macs_per_step"])
print("ACs per step:", mlp_metrics["acs_per_step"])
print("Energy per step (uJ):", mlp_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("")
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", mlp_metrics["binning_latency_ms"])

===== MLP 5M =====
Layer sizes: [5120, 640, 640, 640, 640, 640, 41]

--- Model size ---
Parameters: 4944681
Parameter count matches Optuna reported count: 4944681
Memory footprint (bits): 158229792
Memory footprint (MB): 18.86246109008789

--- Per-step operations ---
MACs per step: 4941440
ACs per step: 3241
Energy per step (uJ): 22.733540899999998

--- Effective sequence length ---
Mean effective length: 218.6107601600337
Median effective length: 209.0
P95 effective length: 358.0

--- Operations per inference ---
Mean MACs: 1080251954.6851969
Median MACs: 1032760960.0
P95 MACs: 1769035520.0
Mean ACs: 708517.4736786692
Median ACs: 677369.0
P95 ACs: 1160278.0

--- Energy per inference ---

Mean energy (uJ): 4969.796657278216
Median energy (uJ): 4751.310048099999
P95 energy (uJ): 8138.607642199999

--- Binning latency ---
Binning latency (ms): 180


## GRU

In [9]:
def raw_to_effective_length(raw_length, patch_size, patch_stride):
    raw_length = int(raw_length)
    if raw_length < patch_size:
        return 0
    return ((raw_length - patch_size) // patch_stride) + 1

def compute_gru_metrics_from_csv(
    raw_lengths_csv,
    d,
    h,
    L,
    K,
    patch_size,
    patch_stride,
):
    df = pd.read_csv(raw_lengths_csv)

    input_params = 3 * h * d + 3 * h
    recurrent_params_first = 3 * h * h + 3 * h

    if L > 1:
        recurrent_params_other_layers = (L - 1) * (6 * h * h + 6 * h)
    else:
        recurrent_params_other_layers = 0

    output_params = K * h + K
    h0_params = h

    n_params = (
        input_params
        + recurrent_params_first
        + recurrent_params_other_layers
        + output_params
        + h0_params
    )

    macs_per_step = (
        3 * h * d
        + (6 * L - 3) * h * h
        + K * h
    )

    acs_per_step = (
        6 * L * h
        + K
    )

    df["effective_length"] = df["raw_length"].apply(
        lambda x: raw_to_effective_length(x, patch_size, patch_stride)
    )

    df["macs"] = df["effective_length"] * macs_per_step
    df["acs"] = df["effective_length"] * acs_per_step
    df["energy_J"] = df["macs"] * ENERGY_MAC + df["acs"] * ENERGY_AC
    df["energy_mJ"] = df["energy_J"] * 1e3
    df["energy_uJ"] = df["energy_J"] * 1e6

    memory_bits = n_params * BITS_PER_PARAM
    memory_bytes = memory_bits / 8
    memory_kb = memory_bytes / 1024
    memory_mb = memory_bytes / 1024**2

    binning_latency_ms = (patch_size - 1) * BIN_SIZE_MS

    aggregate = {
        "mean_effective_length": df["effective_length"].mean(),
        "median_effective_length": df["effective_length"].median(),
        "p95_effective_length": df["effective_length"].quantile(0.95),

        "mean_macs": df["macs"].mean(),
        "median_macs": df["macs"].median(),
        "p95_macs": df["macs"].quantile(0.95),

        "mean_acs": df["acs"].mean(),
        "median_acs": df["acs"].median(),
        "p95_acs": df["acs"].quantile(0.95),

        "mean_energy_J": df["energy_J"].mean(),
        "median_energy_J": df["energy_J"].median(),
        "p95_energy_J": df["energy_J"].quantile(0.95),

        "mean_energy_uJ": df["energy_uJ"].mean(),
        "median_energy_uJ": df["energy_uJ"].median(),
        "p95_energy_uJ": df["energy_uJ"].quantile(0.95),
    }

    return {
        "d": d,
        "h": h,
        "L": L,
        "K": K,

        "n_params": n_params,
        "memory_bits": memory_bits,
        "memory_bytes": memory_bytes,
        "memory_kB": memory_kb,
        "memory_MB": memory_mb,

        "macs_per_step": macs_per_step,
        "acs_per_step": acs_per_step,
        "energy_per_step_J": macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC,
        "energy_per_step_uJ": (macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC) * 1e6,

        "patch_size": patch_size,
        "patch_stride": patch_stride,
        "binning_latency_ms": binning_latency_ms,

        "per_utterance_df": df,
        "aggregate": aggregate,
    }


In [10]:
# ===== GRU CONFIG 0.5M =====

patch_size = 4
patch_stride = 4
input_dim = INPUT_FEATURES * patch_size
h = 64
L = 4
K = OUTPUT_SIZE

gru_metrics = compute_gru_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    d=input_dim,
    h=h,
    L=L,
    K=K,
    patch_size=patch_size,
    patch_stride=patch_stride,
)

agg = gru_metrics["aggregate"]

print("\n===== GRU 0.5M =====")
print("d:", gru_metrics["d"])
print("h:", gru_metrics["h"])
print("L:", gru_metrics["L"])
print("K:", gru_metrics["K"])

print("\n--- Model size ---")
print("Parameters:", gru_metrics["n_params"])
print("Parameter count matches Optuna reported count: 483,497")
print("Memory footprint (bits):", gru_metrics["memory_bits"])
print("Memory footprint (MB):", gru_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per timestep:", gru_metrics["macs_per_step"])
print("ACs per timestep:", gru_metrics["acs_per_step"])
print("Energy per timestep (uJ):", gru_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", gru_metrics["binning_latency_ms"])


===== GRU 0.5M =====
d: 2048
h: 64
L: 4
K: 41

--- Model size ---
Parameters: 483497
Parameter count matches Optuna reported count: 483,497
Memory footprint (bits): 15471904
Memory footprint (MB): 1.8443946838378906

--- Per-step operations ---
MACs per timestep: 481856
ACs per timestep: 1577
Energy per timestep (uJ): 2.2179569

--- Effective sequence length ---
Mean effective length: 220.107180459044
Median effective length: 211.0
P95 effective length: 359.0

--- Operations per inference ---
Mean MACs: 106059965.54727311
Median MACs: 101671616.0
P95 MACs: 172986304.0
Mean ACs: 347109.0235839124
Median ACs: 332747.0
P95 ACs: 566143.0

--- Energy per inference ---
Mean energy (J): 0.00048818823963868186
Mean energy (uJ): 488.18823963868175
Median energy (uJ): 467.98890589999996
P95 energy (uJ): 796.2465271

--- Binning latency ---
Binning latency (ms): 60


In [11]:
# ===== GRU CONFIG 2M =====

patch_size = 4
patch_stride = 4
input_dim = INPUT_FEATURES * patch_size
h = 256
L = 2
K = OUTPUT_SIZE

gru_metrics = compute_gru_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    d=input_dim,
    h=h,
    L=L,
    K=K,
    patch_size=patch_size,
    patch_stride=patch_stride,
)

agg = gru_metrics["aggregate"]

print("\n===== GRU 2M =====")
print("d:", gru_metrics["d"])
print("h:", gru_metrics["h"])
print("L:", gru_metrics["L"])
print("K:", gru_metrics["K"])

print("\n--- Model size ---")
print("Parameters:", gru_metrics["n_params"])
print("Parameter count matches Optuna reported count: 2176553")
print("Memory footprint (bits):", gru_metrics["memory_bits"])
print("Memory footprint (MB):", gru_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per timestep:", gru_metrics["macs_per_step"])
print("ACs per timestep:", gru_metrics["acs_per_step"])
print("Energy per timestep (uJ):", gru_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", gru_metrics["binning_latency_ms"])


===== GRU 2M =====
d: 2048
h: 256
L: 2
K: 41

--- Model size ---
Parameters: 2176553
Parameter count matches Optuna reported count: 2176553
Memory footprint (bits): 69649696
Memory footprint (MB): 8.30289077758789

--- Per-step operations ---
MACs per timestep: 2173184
ACs per timestep: 3113
Energy per timestep (uJ): 9.999448099999999

--- Effective sequence length ---
Mean effective length: 220.107180459044
Median effective length: 211.0
P95 effective length: 359.0

--- Operations per inference ---
Mean MACs: 478333402.8587071
Median MACs: 458541824.0
P95 MACs: 780173056.0
Mean ACs: 685193.652769004
Median ACs: 656843.0
P95 ACs: 1117567.0

--- Energy per inference ---
Mean energy (J): 0.002200950327437545
Mean energy (uJ): 2200.9503274375443
Median energy (uJ): 2109.8835491
P95 energy (uJ): 3589.8018678999997

--- Binning latency ---
Binning latency (ms): 60


In [12]:
# ===== GRU CONFIG 5M =====

patch_size = 10
patch_stride = 4
input_dim = INPUT_FEATURES * patch_size
h = 256
L = 3
K = OUTPUT_SIZE

gru_metrics = compute_gru_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    d=input_dim,
    h=h,
    L=L,
    K=K,
    patch_size=patch_size,
    patch_stride=patch_stride,
)

agg = gru_metrics["aggregate"]

print("\n===== GRU 5M =====")
print("d:", gru_metrics["d"])
print("h:", gru_metrics["h"])
print("L:", gru_metrics["L"])
print("K:", gru_metrics["K"])

print("\n--- Model size ---")
print("Parameters:", gru_metrics["n_params"])
print("Parameter count matches Optuna reported count: 4,930,601")
print("Memory footprint (bits):", gru_metrics["memory_bits"])
print("Memory footprint (MB):", gru_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per timestep:", gru_metrics["macs_per_step"])
print("ACs per timestep:", gru_metrics["acs_per_step"])
print("Energy per timestep (uJ):", gru_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", gru_metrics["binning_latency_ms"])


===== GRU 5M =====
d: 5120
h: 256
L: 3
K: 41

--- Model size ---
Parameters: 4930601
Parameter count matches Optuna reported count: 4,930,601
Memory footprint (bits): 157779232
Memory footprint (MB): 18.80875015258789

--- Per-step operations ---
MACs per timestep: 4925696
ACs per timestep: 4649
Energy per timestep (uJ): 22.662385699999998

--- Effective sequence length ---
Mean effective length: 218.6107601600337
Median effective length: 209.0
P95 effective length: 358.0

--- Operations per inference ---
Mean MACs: 1076810146.8772373
Median MACs: 1029470464.0
P95 MACs: 1763399168.0
Mean ACs: 1016321.4239839966
Median ACs: 971641.0
P95 ACs: 1664342.0

--- Energy per inference ---
Mean energy (J): 0.004954241364916878
Mean energy (uJ): 4954.2413649168775
Median energy (uJ): 4736.4386113
P95 energy (uJ): 8113.1340806

--- Binning latency ---
Binning latency (ms): 180


## LSTM

In [13]:
def raw_to_effective_length(raw_length, patch_size, patch_stride):
    raw_length = int(raw_length)
    if raw_length < patch_size:
        return 0
    return ((raw_length - patch_size) // patch_stride) + 1

def compute_lstm_metrics_from_csv(
    raw_lengths_csv,
    d,
    h,
    L,
    K,
    patch_size,
    patch_stride,
):
    df = pd.read_csv(raw_lengths_csv)

    input_params_first = 4 * h * d
    recurrent_params_first = 4 * h * h
    bias_params_first = 8 * h

    if L > 1:
        other_layer_params = (L - 1) * (8 * h * h + 8 * h)
    else:
        other_layer_params = 0

    output_params = K * h + K

    learned_initial_state_params = 2 * h

    n_params = (
        input_params_first
        + recurrent_params_first
        + bias_params_first
        + other_layer_params
        + output_params
        + learned_initial_state_params
    )

    macs_per_step = (
        4 * h * d
        + (8 * L - 4) * h * h
        + K * h
    )

    acs_per_step = (
        8 * L * h
        + K
    )

    df["effective_length"] = df["raw_length"].apply(
        lambda x: raw_to_effective_length(x, patch_size, patch_stride)
    )

    df["macs"] = df["effective_length"] * macs_per_step
    df["acs"] = df["effective_length"] * acs_per_step
    df["energy_J"] = df["macs"] * ENERGY_MAC + df["acs"] * ENERGY_AC
    df["energy_mJ"] = df["energy_J"] * 1e3
    df["energy_uJ"] = df["energy_J"] * 1e6

    memory_bits = n_params * BITS_PER_PARAM
    memory_bytes = memory_bits / 8
    memory_kb = memory_bytes / 1024
    memory_mb = memory_bytes / 1024**2

    binning_latency_ms = (patch_size - 1) * BIN_SIZE_MS

    aggregate = {
        "mean_effective_length": df["effective_length"].mean(),
        "median_effective_length": df["effective_length"].median(),
        "p95_effective_length": df["effective_length"].quantile(0.95),

        "mean_macs": df["macs"].mean(),
        "median_macs": df["macs"].median(),
        "p95_macs": df["macs"].quantile(0.95),

        "mean_acs": df["acs"].mean(),
        "median_acs": df["acs"].median(),
        "p95_acs": df["acs"].quantile(0.95),

        "mean_energy_J": df["energy_J"].mean(),
        "median_energy_J": df["energy_J"].median(),
        "p95_energy_J": df["energy_J"].quantile(0.95),

        "mean_energy_uJ": df["energy_uJ"].mean(),
        "median_energy_uJ": df["energy_uJ"].median(),
        "p95_energy_uJ": df["energy_uJ"].quantile(0.95),
    }

    return {
        "d": d,
        "h": h,
        "L": L,
        "K": K,

        "n_params": n_params,
        "memory_bits": memory_bits,
        "memory_bytes": memory_bytes,
        "memory_kB": memory_kb,
        "memory_MB": memory_mb,

        "macs_per_step": macs_per_step,
        "acs_per_step": acs_per_step,
        "energy_per_step_J": macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC,
        "energy_per_step_uJ": (macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC) * 1e6,

        "patch_size": patch_size,
        "patch_stride": patch_stride,
        "binning_latency_ms": binning_latency_ms,

        "per_utterance_df": df,
        "aggregate": aggregate,
    }


In [14]:
# ===== LSTM CONFIG 0.5M =====

patch_size = 8
d = INPUT_FEATURES * patch_size
h = 32
L = 3
K = OUTPUT_SIZE

lstm_metrics = compute_lstm_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    d=d,
    h=h,
    L=L,
    K=K,
    patch_size=patch_size,
    patch_stride=PATCH_STRIDE,
)

agg = lstm_metrics["aggregate"]

print("\n===== LSTM 0.5M =====")
print("d:", lstm_metrics["d"])
print("h:", lstm_metrics["h"])
print("L:", lstm_metrics["L"])
print("K:", lstm_metrics["K"])

print("\n--- Model size ---")
print("Parameters:", lstm_metrics["n_params"])
print("Parameter count matches Optuna reported count: 546,953")
print("Memory footprint (bits):", lstm_metrics["memory_bits"])
print("Memory footprint (MB):", lstm_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per timestep:", lstm_metrics["macs_per_step"])
print("ACs per timestep:", lstm_metrics["acs_per_step"])
print("Energy per timestep (uJ):", lstm_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", lstm_metrics["binning_latency_ms"])


===== LSTM 0.5M =====
d: 4096
h: 32
L: 3
K: 41

--- Model size ---
Parameters: 546953
Parameter count matches Optuna reported count: 546,953
Memory footprint (bits): 17502496
Memory footprint (MB): 2.0864601135253906

--- Per-step operations ---
MACs per timestep: 546080
ACs per timestep: 809
Energy per timestep (uJ): 2.5126961

--- Effective sequence length ---
Mean effective length: 219.107180459044
Median effective length: 210.0
P95 effective length: 358.0

--- Operations per inference ---
Mean MACs: 119650049.10507475
Median MACs: 114676800.0
P95 MACs: 195496640.0
Mean ACs: 177257.7089913666
Median ACs: 169890.0
P95 ACs: 289622.0

--- Energy per inference ---
Mean energy (J): 0.000550549757821436
Mean energy (uJ): 550.5497578214361
Median energy (uJ): 527.6661809999999
P95 energy (uJ): 899.5452038

--- Binning latency ---
Binning latency (ms): 140


In [15]:
# ===== LSTM CONFIG 2M =====

patch_size = 8
d = INPUT_FEATURES * patch_size
h = 128
L = 2
K = OUTPUT_SIZE

lstm_metrics = compute_lstm_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    d=d,
    h=h,
    L=L,
    K=K,
    patch_size=patch_size,
    patch_stride=PATCH_STRIDE,
)

agg = lstm_metrics["aggregate"]

print("\n===== LSTM 2M =====")
print("d:", lstm_metrics["d"])
print("h:", lstm_metrics["h"])
print("L:", lstm_metrics["L"])
print("K:", lstm_metrics["K"])

print("\n--- Model size ---")
print("Parameters:", lstm_metrics["n_params"])
print("Parameter count matches Optuna reported count: 2301353")
print("Memory footprint (bits):", lstm_metrics["memory_bits"])
print("Memory footprint (MB):", lstm_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per timestep:", lstm_metrics["macs_per_step"])
print("ACs per timestep:", lstm_metrics["acs_per_step"])
print("Energy per timestep (uJ):", lstm_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", lstm_metrics["binning_latency_ms"])


===== LSTM 2M =====
d: 4096
h: 128
L: 2
K: 41

--- Model size ---
Parameters: 2301353
Parameter count matches Optuna reported count: 2301353
Memory footprint (bits): 73643296
Memory footprint (MB): 8.77896499633789

--- Per-step operations ---
MACs per timestep: 2299008
ACs per timestep: 2089
Energy per timestep (uJ): 10.5773169

--- Effective sequence length ---
Mean effective length: 219.107180459044
Median effective length: 210.0
P95 effective length: 358.0

--- Operations per inference ---
Mean MACs: 503729160.7327858
Median MACs: 482791680.0
P95 MACs: 823044864.0
Mean ACs: 457714.89997894294
Median ACs: 438690.0
P95 ACs: 747862.0

--- Energy per inference ---
Mean energy (J): 0.0023175660827807957
Mean energy (uJ): 2317.5660827807956
Median energy (uJ): 2221.236549
P95 energy (uJ): 3786.6794502

--- Binning latency ---
Binning latency (ms): 140


In [16]:
# ===== LSTM CONFIG 5M =====

patch_size = 6
d = INPUT_FEATURES * patch_size
h = 256
L = 4
K = OUTPUT_SIZE

lstm_metrics = compute_lstm_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    d=d,
    h=h,
    L=L,
    K=K,
    patch_size=patch_size,
    patch_stride=PATCH_STRIDE,
)

agg = lstm_metrics["aggregate"]

print("\n===== LSTM 5M =====")
print("d:", lstm_metrics["d"])
print("h:", lstm_metrics["h"])
print("L:", lstm_metrics["L"])
print("K:", lstm_metrics["K"])

print("\n--- Model size ---")
print("Parameters:", lstm_metrics["n_params"])
print("Parameter count matches Optuna reported count: 4,999,977")
print("Memory footprint (bits):", lstm_metrics["memory_bits"])
print("Memory footprint (MB):", lstm_metrics["memory_MB"])

print("\n--- Per-step operations ---")
print("MACs per timestep:", lstm_metrics["macs_per_step"])
print("ACs per timestep:", lstm_metrics["acs_per_step"])
print("Energy per timestep (uJ):", lstm_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", lstm_metrics["binning_latency_ms"])


===== LSTM 5M =====
d: 3072
h: 256
L: 4
K: 41

--- Model size ---
Parameters: 4999977
Parameter count matches Optuna reported count: 4,999,977
Memory footprint (bits): 159999264
Memory footprint (MB): 19.07339859008789

--- Per-step operations ---
MACs per timestep: 4991232
ACs per timestep: 8233
Energy per timestep (uJ): 22.9670769

--- Effective sequence length ---
Mean effective length: 219.6107601600337
Median effective length: 210.0
P95 effective length: 359.0

--- Operations per inference ---
Mean MACs: 1096128253.6550853
Median MACs: 1048158720.0
P95 MACs: 1791852288.0
Mean ACs: 1808055.3883975574
Median ACs: 1728930.0
P95 ACs: 2955647.0

--- Energy per inference ---
Mean energy (J): 0.00504381721666295
Mean energy (uJ): 5043.817216662949
Median energy (uJ): 4823.086149
P95 energy (uJ): 8245.1806071

--- Binning latency ---
Binning latency (ms): 100


## SNN

In [17]:
def raw_to_effective_length(raw_length, patch_size, patch_stride):
    if raw_length < patch_size:
        return 0
    return (raw_length - patch_size) // patch_stride + 1


def compute_snn_metrics_from_csv(
    raw_lengths_csv,
    input_features,
    output_size,
    hidden_size,
    n_blocks,
    patch_size,
    patch_stride,
    mean_spike_rates,   # list/tuple, one rho per block
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=False,
    print_breakdown=True,
):
    D = input_features
    K = output_size
    h = hidden_size
    B = n_blocks
    P = patch_size

    if len(mean_spike_rates) != B:
        raise ValueError(
            f"Expected {B} spike rates, got {len(mean_spike_rates)}."
        )

    rhos = [float(rho) for rho in mean_spike_rates]

    df = pd.read_csv(raw_lengths_csv)
    df["effective_length"] = df["raw_length"].apply(
        lambda x: raw_to_effective_length(x, patch_size, patch_stride)
    )

    input_temporal_params = D * P + D
    input_temporal_macs_per_step = D * P
    input_temporal_acs_per_step = D

    input_expand_params = D * h + h
    input_expand_macs_per_step = D * h
    input_expand_acs_per_step = 0

    input_params = input_temporal_params + input_expand_params
    input_macs_per_step = input_temporal_macs_per_step + input_expand_macs_per_step
    input_acs_per_step = h

    lagged_params_per_block = h * h + h
    lif_params_per_block = 2
    bn_params_per_block = 2 * h if include_bn_params else 0

    block_params_per_block = (
        lagged_params_per_block
        + lif_params_per_block
        + bn_params_per_block
    )
    block_params = B * block_params_per_block

    block_lif_acs_per_step = B * c_lif * h

    block_delay_acs_per_step_by_block = [
        rho * h * h + rho * h
        for rho in rhos
    ]

    block_delay_acs_per_step = sum(block_delay_acs_per_step_by_block)

    block_bn_acs_per_step = B * (4 * h if include_bn_ops else 0)
    block_residual_acs_per_step = B * h

    block_macs_per_step = 0
    block_acs_per_step = (
        block_lif_acs_per_step
        + block_delay_acs_per_step
        + block_bn_acs_per_step
        + block_residual_acs_per_step
    )

    readout_first_params = h * h + h
    readout_output_params = h * K + K
    readout_params = readout_first_params + readout_output_params

    readout_macs_per_step = h * h + h * K
    readout_acs_per_step = (h + K) + (h if include_gelu_ops else 0)

    n_params = input_params + block_params + readout_params

    macs_per_step = input_macs_per_step + block_macs_per_step + readout_macs_per_step
    acs_per_step = input_acs_per_step + block_acs_per_step + readout_acs_per_step

    df["macs"] = df["effective_length"] * macs_per_step
    df["acs"] = df["effective_length"] * acs_per_step
    df["energy_J"] = df["macs"] * ENERGY_MAC + df["acs"] * ENERGY_AC
    df["energy_uJ"] = df["energy_J"] * 1e6

    component_ops_per_step = {
        "input_temporal_projection": {
            "macs": input_temporal_macs_per_step,
            "acs": input_temporal_acs_per_step,
        },
        "input_expansion": {
            "macs": input_expand_macs_per_step,
            "acs": input_expand_acs_per_step,
        },
        "block_lif": {
            "macs": 0,
            "acs": block_lif_acs_per_step,
        },
        "block_delays_total": {
            "macs": 0,
            "acs": block_delay_acs_per_step,
        },
        "block_batchnorm": {
            "macs": 0,
            "acs": block_bn_acs_per_step,
        },
        "block_residual": {
            "macs": 0,
            "acs": block_residual_acs_per_step,
        },
        "readout": {
            "macs": readout_macs_per_step,
            "acs": readout_acs_per_step,
        },
    }

    for i, delay_acs in enumerate(block_delay_acs_per_step_by_block):
        component_ops_per_step[f"block_{i}_delay"] = {
            "macs": 0,
            "acs": delay_acs,
        }

    component_breakdown = {}

    for name, ops in component_ops_per_step.items():
        macs_step = ops["macs"]
        acs_step = ops["acs"]

        df[f"{name}_macs"] = df["effective_length"] * macs_step
        df[f"{name}_acs"] = df["effective_length"] * acs_step
        df[f"{name}_energy_J"] = (
            df[f"{name}_macs"] * ENERGY_MAC
            + df[f"{name}_acs"] * ENERGY_AC
        )
        df[f"{name}_energy_uJ"] = df[f"{name}_energy_J"] * 1e6

        component_breakdown[name] = {
            "macs_per_step": macs_step,
            "acs_per_step": acs_step,
            "mean_macs": df[f"{name}_macs"].mean(),
            "mean_acs": df[f"{name}_acs"].mean(),
            "mean_energy_uJ": df[f"{name}_energy_uJ"].mean(),
        }

    memory_bits = n_params * BITS_PER_PARAM
    memory_bytes = memory_bits / 8
    memory_MB = memory_bytes / 1024**2

    binning_latency_ms = (patch_size - 1) * BIN_SIZE_MS

    aggregate = {
        "mean_effective_length": df["effective_length"].mean(),
        "median_effective_length": df["effective_length"].median(),
        "p95_effective_length": df["effective_length"].quantile(0.95),

        "mean_macs": df["macs"].mean(),
        "median_macs": df["macs"].median(),
        "p95_macs": df["macs"].quantile(0.95),

        "mean_acs": df["acs"].mean(),
        "median_acs": df["acs"].median(),
        "p95_acs": df["acs"].quantile(0.95),

        "mean_energy_J": df["energy_J"].mean(),
        "median_energy_J": df["energy_J"].median(),
        "p95_energy_J": df["energy_J"].quantile(0.95),

        "mean_energy_uJ": df["energy_uJ"].mean(),
        "median_energy_uJ": df["energy_uJ"].median(),
        "p95_energy_uJ": df["energy_uJ"].quantile(0.95),
    }

    if print_breakdown:
        print("\nEnergy breakdown, mean per inference:")
        for name, values in component_breakdown.items():
            print(
                f"{name:30s} "
                f"MACs={values['mean_macs']:.3e} "
                f"ACs={values['mean_acs']:.3e} "
                f"Energy={values['mean_energy_uJ']:.3f} uJ"
            )

        print(
            f"{'TOTAL':30s} "
            f"MACs={aggregate['mean_macs']:.3e} "
            f"ACs={aggregate['mean_acs']:.3e} "
            f"Energy={aggregate['mean_energy_uJ']:.3f} uJ"
        )

    return {
        "input_mode": "depthwise_temporal_plus_dense_expansion",

        "D": D,
        "P": P,
        "h": h,
        "B": B,
        "K": K,
        "mean_spike_rates": rhos,

        "input_temporal_params": input_temporal_params,
        "input_expand_params": input_expand_params,
        "input_params": input_params,
        "block_params": block_params,
        "readout_params": readout_params,
        "n_params": n_params,

        "memory_bits": memory_bits,
        "memory_bytes": memory_bytes,
        "memory_MB": memory_MB,

        "input_temporal_macs_per_step": input_temporal_macs_per_step,
        "input_expand_macs_per_step": input_expand_macs_per_step,
        "input_macs_per_step": input_macs_per_step,
        "input_acs_per_step": input_acs_per_step,

        "block_macs_per_step": block_macs_per_step,
        "block_acs_per_step": block_acs_per_step,

        "block_delay_acs_per_step_by_block": block_delay_acs_per_step_by_block,
        "block_delay_acs_per_step": block_delay_acs_per_step,

        "block_lif_acs_per_step": block_lif_acs_per_step,
        "block_bn_acs_per_step": block_bn_acs_per_step,
        "block_residual_acs_per_step": block_residual_acs_per_step,

        "readout_macs_per_step": readout_macs_per_step,
        "readout_acs_per_step": readout_acs_per_step,

        "macs_per_step": macs_per_step,
        "acs_per_step": acs_per_step,

        "energy_per_step_J": macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC,
        "energy_per_step_uJ": (
            macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC
        ) * 1e6,

        "patch_size": patch_size,
        "patch_stride": patch_stride,
        "binning_latency_ms": binning_latency_ms,

        "component_breakdown": component_breakdown,
        "per_utterance_df": df,
        "aggregate": aggregate,
    }

In [18]:
metrics = compute_snn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=256,
    n_blocks=4,
    patch_size=6,
    patch_stride=PATCH_STRIDE,
    mean_spike_rates=[0.10, 0.13, 0.16, 0.19],
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=True,
    print_breakdown=True,
)

print("\nSummary SNN 0.5M:")
print(f"Params: {metrics['n_params']:,}")
print(f"Storage: {metrics['memory_MB']:.2f} MB")
print(f"Mean MACs: {metrics['aggregate']['mean_macs'] / 1e6:.2f}M")
print(f"Mean ACs: {metrics['aggregate']['mean_acs'] / 1e6:.2f}M")
print(f"Mean energy: {metrics['aggregate']['mean_energy_uJ']:.2f} uJ")
print(f"Binning latency: {metrics['binning_latency_ms']} ms")


Energy breakdown, mean per inference:
input_temporal_projection      MACs=6.746e+05 ACs=1.124e+05 Energy=3.205 uJ
input_expansion                MACs=2.878e+07 ACs=0.000e+00 Energy=132.410 uJ
block_lif                      MACs=0.000e+00 ACs=8.995e+05 Energy=0.810 uJ
block_delays_total             MACs=0.000e+00 ACs=8.380e+06 Energy=7.542 uJ
block_batchnorm                MACs=0.000e+00 ACs=8.995e+05 Energy=0.810 uJ
block_residual                 MACs=0.000e+00 ACs=2.249e+05 Energy=0.202 uJ
readout                        MACs=1.670e+07 ACs=1.214e+05 Energy=76.918 uJ
block_0_delay                  MACs=0.000e+00 ACs=1.445e+06 Energy=1.300 uJ
block_1_delay                  MACs=0.000e+00 ACs=1.878e+06 Energy=1.690 uJ
block_2_delay                  MACs=0.000e+00 ACs=2.312e+06 Energy=2.081 uJ
block_3_delay                  MACs=0.000e+00 ACs=2.745e+06 Energy=2.471 uJ
TOTAL                          MACs=4.616e+07 ACs=1.058e+07 Energy=221.845 uJ

Summary SNN 0.5M:
Params: 476,465
Storage: 

In [19]:
metrics = compute_snn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=512,
    n_blocks=4,
    patch_size=6,
    patch_stride=PATCH_STRIDE,
    mean_spike_rates=[0.10, 0.13, 0.16, 0.19],
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=True,
    print_breakdown=True,
)

print("\nSummary SNN 2M:")
print(f"Params: {metrics['n_params']:,}")
print(f"Storage: {metrics['memory_MB']:.2f} MB")
print(f"Mean MACs: {metrics['aggregate']['mean_macs'] / 1e6:.2f}M")
print(f"Mean ACs: {metrics['aggregate']['mean_acs'] / 1e6:.2f}M")
print(f"Mean energy: {metrics['aggregate']['mean_energy_uJ']:.2f} uJ")
print(f"Binning latency: {metrics['binning_latency_ms']} ms")


Energy breakdown, mean per inference:
input_temporal_projection      MACs=6.746e+05 ACs=1.124e+05 Energy=3.205 uJ
input_expansion                MACs=5.757e+07 ACs=0.000e+00 Energy=264.820 uJ
block_lif                      MACs=0.000e+00 ACs=1.799e+06 Energy=1.619 uJ
block_delays_total             MACs=0.000e+00 ACs=3.346e+07 Energy=30.110 uJ
block_batchnorm                MACs=0.000e+00 ACs=1.799e+06 Energy=1.619 uJ
block_residual                 MACs=0.000e+00 ACs=4.498e+05 Energy=0.405 uJ
readout                        MACs=6.218e+07 ACs=2.339e+05 Energy=286.237 uJ
block_0_delay                  MACs=0.000e+00 ACs=5.768e+06 Energy=5.191 uJ
block_1_delay                  MACs=0.000e+00 ACs=7.499e+06 Energy=6.749 uJ
block_2_delay                  MACs=0.000e+00 ACs=9.229e+06 Energy=8.306 uJ
block_3_delay                  MACs=0.000e+00 ACs=1.096e+07 Energy=9.864 uJ
TOTAL                          MACs=1.204e+08 ACs=3.785e+07 Energy=588.015 uJ

Summary SNN 2M:
Params: 1,604,657
Storage

In [20]:
metrics = compute_snn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=800,      # now this can be >512
    n_blocks=5,
    patch_size=6,
    patch_stride=PATCH_STRIDE,
    mean_spike_rates=[0.07, 0.18, 0.23, 0.26, 0.28],
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=True,
    print_breakdown=True,
)

print("\nSummary SNN 5M:")
print(f"Params: {metrics['n_params']:,}")
print(f"Storage: {metrics['memory_MB']:.2f} MB")
print(f"Mean MACs: {metrics['aggregate']['mean_macs'] / 1e6:.2f}M")
print(f"Mean ACs: {metrics['aggregate']['mean_acs'] / 1e6:.2f}M")
print(f"Mean energy / inference: {metrics['aggregate']['mean_energy_uJ']:.2f} uJ")
print(f"Binning latency: {metrics['binning_latency_ms']} ms")


Energy breakdown, mean per inference:
input_temporal_projection      MACs=6.746e+05 ACs=1.124e+05 Energy=3.205 uJ
input_expansion                MACs=8.995e+07 ACs=0.000e+00 Energy=413.782 uJ
block_lif                      MACs=0.000e+00 ACs=3.514e+06 Energy=3.162 uJ
block_delays_total             MACs=0.000e+00 ACs=1.435e+08 Energy=129.187 uJ
block_batchnorm                MACs=0.000e+00 ACs=3.514e+06 Energy=3.162 uJ
block_residual                 MACs=0.000e+00 ACs=8.784e+05 Energy=0.791 uJ
readout                        MACs=1.478e+08 ACs=3.604e+05 Energy=679.993 uJ
block_0_delay                  MACs=0.000e+00 ACs=9.851e+06 Energy=8.866 uJ
block_1_delay                  MACs=0.000e+00 ACs=2.533e+07 Energy=22.798 uJ
block_2_delay                  MACs=0.000e+00 ACs=3.237e+07 Energy=29.130 uJ
block_3_delay                  MACs=0.000e+00 ACs=3.659e+07 Energy=32.930 uJ
block_4_delay                  MACs=0.000e+00 ACs=3.940e+07 Energy=35.463 uJ
TOTAL                          MACs=2.3

In [30]:
# No delays: delay mechanism removed from calculations,
# but mean_spike_rates kept for reporting.

def raw_to_effective_length(raw_length, patch_size, patch_stride):
    if raw_length < patch_size:
        return 0
    return (raw_length - patch_size) // patch_stride + 1


def compute_snn_metrics_from_csv(
    raw_lengths_csv,
    input_features,
    output_size,
    hidden_size,
    n_blocks,
    patch_size,
    patch_stride,
    mean_spike_rates=None,   # kept for reporting only
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=False,
    print_breakdown=True,
):
    D = input_features
    K = output_size
    h = hidden_size
    B = n_blocks
    P = patch_size

    if mean_spike_rates is None:
        rhos = None
    else:
        if len(mean_spike_rates) != B:
            raise ValueError(
                f"Expected {B} spike rates, got {len(mean_spike_rates)}."
            )
        rhos = [float(rho) for rho in mean_spike_rates]

    df = pd.read_csv(raw_lengths_csv)
    df["effective_length"] = df["raw_length"].apply(
        lambda x: raw_to_effective_length(x, patch_size, patch_stride)
    )

    # ------------------------------------------------------------
    # Input stage:
    # Depthwise temporal projection: [D * P] -> [D]
    # Dense expansion: [D] -> [h]
    # ------------------------------------------------------------
    input_temporal_params = D * P + D
    input_temporal_macs_per_step = D * P
    input_temporal_acs_per_step = D

    input_expand_params = D * h + h
    input_expand_macs_per_step = D * h
    input_expand_acs_per_step = h

    input_params = input_temporal_params + input_expand_params
    input_macs_per_step = input_temporal_macs_per_step + input_expand_macs_per_step
    input_acs_per_step = input_temporal_acs_per_step + input_expand_acs_per_step

    # ------------------------------------------------------------
    # Residual SNN blocks without delay mechanism.
    # Count only LIF + optional BN + residual addition.
    # No delayed mixer parameters or delayed mixer operations.
    # ------------------------------------------------------------
    lif_params_per_block = 2
    bn_params_per_block = 2 * h if include_bn_params else 0

    block_params_per_block = lif_params_per_block + bn_params_per_block
    block_params = B * block_params_per_block

    block_lif_acs_per_step = B * c_lif * h
    block_bn_acs_per_step = B * (4 * h if include_bn_ops else 0)
    block_residual_acs_per_step = B * h

    block_macs_per_step = 0
    block_acs_per_step = (
        block_lif_acs_per_step
        + block_bn_acs_per_step
        + block_residual_acs_per_step
    )

    # ------------------------------------------------------------
    # Readout:
    # Linear h -> h, GELU, dropout, Linear h -> K
    # ------------------------------------------------------------
    readout_first_params = h * h + h
    readout_output_params = h * K + K
    readout_params = readout_first_params + readout_output_params

    readout_macs_per_step = h * h + h * K
    readout_acs_per_step = (h + K) + (h if include_gelu_ops else 0)

    # ------------------------------------------------------------
    # Totals
    # ------------------------------------------------------------
    n_params = input_params + block_params + readout_params

    macs_per_step = (
        input_macs_per_step
        + block_macs_per_step
        + readout_macs_per_step
    )

    acs_per_step = (
        input_acs_per_step
        + block_acs_per_step
        + readout_acs_per_step
    )

    df["macs"] = df["effective_length"] * macs_per_step
    df["acs"] = df["effective_length"] * acs_per_step
    df["energy_J"] = df["macs"] * ENERGY_MAC + df["acs"] * ENERGY_AC
    df["energy_uJ"] = df["energy_J"] * 1e6

    component_ops_per_step = {
        "input_temporal_projection": {
            "macs": input_temporal_macs_per_step,
            "acs": input_temporal_acs_per_step,
        },
        "input_expansion": {
            "macs": input_expand_macs_per_step,
            "acs": input_expand_acs_per_step,
        },
        "block_lif": {
            "macs": 0,
            "acs": block_lif_acs_per_step,
        },
        "block_batchnorm": {
            "macs": 0,
            "acs": block_bn_acs_per_step,
        },
        "block_residual": {
            "macs": 0,
            "acs": block_residual_acs_per_step,
        },
        "readout": {
            "macs": readout_macs_per_step,
            "acs": readout_acs_per_step,
        },
    }

    component_breakdown = {}

    for name, ops in component_ops_per_step.items():
        macs_step = ops["macs"]
        acs_step = ops["acs"]

        df[f"{name}_macs"] = df["effective_length"] * macs_step
        df[f"{name}_acs"] = df["effective_length"] * acs_step
        df[f"{name}_energy_J"] = (
            df[f"{name}_macs"] * ENERGY_MAC
            + df[f"{name}_acs"] * ENERGY_AC
        )
        df[f"{name}_energy_uJ"] = df[f"{name}_energy_J"] * 1e6

        component_breakdown[name] = {
            "macs_per_step": macs_step,
            "acs_per_step": acs_step,
            "mean_macs": df[f"{name}_macs"].mean(),
            "mean_acs": df[f"{name}_acs"].mean(),
            "mean_energy_uJ": df[f"{name}_energy_uJ"].mean(),
        }

    memory_bits = n_params * BITS_PER_PARAM
    memory_bytes = memory_bits / 8
    memory_MB = memory_bytes / 1024**2

    binning_latency_ms = (patch_size - 1) * BIN_SIZE_MS

    aggregate = {
        "mean_effective_length": df["effective_length"].mean(),
        "median_effective_length": df["effective_length"].median(),
        "p95_effective_length": df["effective_length"].quantile(0.95),

        "mean_macs": df["macs"].mean(),
        "median_macs": df["macs"].median(),
        "p95_macs": df["macs"].quantile(0.95),

        "mean_acs": df["acs"].mean(),
        "median_acs": df["acs"].median(),
        "p95_acs": df["acs"].quantile(0.95),

        "mean_energy_J": df["energy_J"].mean(),
        "median_energy_J": df["energy_J"].median(),
        "p95_energy_J": df["energy_J"].quantile(0.95),

        "mean_energy_uJ": df["energy_uJ"].mean(),
        "median_energy_uJ": df["energy_uJ"].median(),
        "p95_energy_uJ": df["energy_uJ"].quantile(0.95),
    }

    if print_breakdown:
        print("\nEnergy breakdown, mean per inference:")
        for name, values in component_breakdown.items():
            print(
                f"{name:30s} "
                f"MACs={values['mean_macs']:.3e} "
                f"ACs={values['mean_acs']:.3e} "
                f"Energy={values['mean_energy_uJ']:.3f} uJ"
            )

        print(
            f"{'TOTAL':30s} "
            f"MACs={aggregate['mean_macs']:.3e} "
            f"ACs={aggregate['mean_acs']:.3e} "
            f"Energy={aggregate['mean_energy_uJ']:.3f} uJ"
        )

    return {
        "input_mode": "depthwise_temporal_plus_dense_expansion_no_delay_count",

        "D": D,
        "P": P,
        "h": h,
        "B": B,
        "K": K,
        "mean_spike_rates": rhos,

        "input_temporal_params": input_temporal_params,
        "input_expand_params": input_expand_params,
        "input_params": input_params,
        "block_params": block_params,
        "readout_params": readout_params,
        "n_params": n_params,

        "memory_bits": memory_bits,
        "memory_bytes": memory_bytes,
        "memory_MB": memory_MB,

        "input_temporal_macs_per_step": input_temporal_macs_per_step,
        "input_expand_macs_per_step": input_expand_macs_per_step,
        "input_macs_per_step": input_macs_per_step,
        "input_acs_per_step": input_acs_per_step,

        "block_macs_per_step": block_macs_per_step,
        "block_acs_per_step": block_acs_per_step,

        "block_lif_acs_per_step": block_lif_acs_per_step,
        "block_bn_acs_per_step": block_bn_acs_per_step,
        "block_residual_acs_per_step": block_residual_acs_per_step,

        "readout_macs_per_step": readout_macs_per_step,
        "readout_acs_per_step": readout_acs_per_step,

        "macs_per_step": macs_per_step,
        "acs_per_step": acs_per_step,

        "energy_per_step_J": macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC,
        "energy_per_step_uJ": (
            macs_per_step * ENERGY_MAC + acs_per_step * ENERGY_AC
        ) * 1e6,

        "patch_size": patch_size,
        "patch_stride": patch_stride,
        "binning_latency_ms": binning_latency_ms,

        "component_breakdown": component_breakdown,
        "per_utterance_df": df,
        "aggregate": aggregate,
    }

In [31]:
metrics = compute_snn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=256,
    n_blocks=4,
    patch_size=6,
    patch_stride=PATCH_STRIDE,
    mean_spike_rates=[0.10, 0.13, 0.16, 0.19],
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=True,
    print_breakdown=True,
)

print("\nSummary SNN 0.5M:")
print(f"Params: {metrics['n_params']:,}")
print(f"Storage: {metrics['memory_MB']:.2f} MB")
print(f"Mean MACs: {metrics['aggregate']['mean_macs'] / 1e6:.4f}M")
print(f"Mean ACs: {metrics['aggregate']['mean_acs'] / 1e6:.4f}M")
print(f"Mean energy: {metrics['aggregate']['mean_energy_uJ']:.4f} uJ")
print(f"Binning latency: {metrics['binning_latency_ms']} ms")


Energy breakdown, mean per inference:
input_temporal_projection      MACs=6.746e+05 ACs=1.124e+05 Energy=3.205 uJ
input_expansion                MACs=2.878e+07 ACs=5.622e+04 Energy=132.461 uJ
block_lif                      MACs=0.000e+00 ACs=8.995e+05 Energy=0.810 uJ
block_batchnorm                MACs=0.000e+00 ACs=8.995e+05 Energy=0.810 uJ
block_residual                 MACs=0.000e+00 ACs=2.249e+05 Energy=0.202 uJ
readout                        MACs=1.670e+07 ACs=1.214e+05 Energy=76.918 uJ
TOTAL                          MACs=4.616e+07 ACs=2.314e+06 Energy=214.404 uJ

Summary SNN 0.5M:
Params: 213,297
Storage: 0.81 MB
Mean MACs: 46.1569M
Mean ACs: 2.3140M
Mean energy: 214.4044 uJ
Binning latency: 100 ms


In [32]:
metrics = compute_snn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=512,
    n_blocks=4,
    patch_size=6,
    patch_stride=PATCH_STRIDE,
    mean_spike_rates=[0.10, 0.13, 0.16, 0.19],
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=True,
    print_breakdown=True,
)

print("\nSummary SNN 2M:")
print(f"Params: {metrics['n_params']:,}")
print(f"Storage: {metrics['memory_MB']:.2f} MB")
print(f"Mean MACs: {metrics['aggregate']['mean_macs'] / 1e6:.4f}M")
print(f"Mean ACs: {metrics['aggregate']['mean_acs'] / 1e6:.4f}M")
print(f"Mean energy: {metrics['aggregate']['mean_energy_uJ']:.4f} uJ")
print(f"Binning latency: {metrics['binning_latency_ms']} ms")


Energy breakdown, mean per inference:
input_temporal_projection      MACs=6.746e+05 ACs=1.124e+05 Energy=3.205 uJ
input_expansion                MACs=5.757e+07 ACs=1.124e+05 Energy=264.922 uJ
block_lif                      MACs=0.000e+00 ACs=1.799e+06 Energy=1.619 uJ
block_batchnorm                MACs=0.000e+00 ACs=1.799e+06 Energy=1.619 uJ
block_residual                 MACs=0.000e+00 ACs=4.498e+05 Energy=0.405 uJ
readout                        MACs=6.218e+07 ACs=2.339e+05 Energy=286.237 uJ
TOTAL                          MACs=1.204e+08 ACs=4.507e+06 Energy=558.006 uJ

Summary SNN 2M:
Params: 554,033
Storage: 2.11 MB
Mean MACs: 120.4240M
Mean ACs: 4.5066M
Mean energy: 558.0064 uJ
Binning latency: 100 ms


In [34]:
metrics = compute_snn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=800,      # now this can be >512
    n_blocks=5,
    patch_size=6,
    patch_stride=PATCH_STRIDE,
    mean_spike_rates=[0.07, 0.18, 0.23, 0.26, 0.28],
    c_lif=4,
    include_bn_params=True,
    include_bn_ops=True,
    include_gelu_ops=True,
    print_breakdown=True,
)
print("\nSummary SNN 5M:")
print(f"Params: {metrics['n_params']:,}")
print(f"Storage: {metrics['memory_MB']:.2f} MB")
print(f"Mean MACs: {metrics['aggregate']['mean_macs'] / 1e6:.4f}M")
print(f"Mean ACs: {metrics['aggregate']['mean_acs'] / 1e6:.4f}M")
print(f"Mean energy: {metrics['aggregate']['mean_energy_uJ']:.4f} uJ")
print(f"Binning latency: {metrics['binning_latency_ms']} ms")


Energy breakdown, mean per inference:
input_temporal_projection      MACs=6.746e+05 ACs=1.124e+05 Energy=3.205 uJ
input_expansion                MACs=8.995e+07 ACs=1.757e+05 Energy=413.940 uJ
block_lif                      MACs=0.000e+00 ACs=3.514e+06 Energy=3.162 uJ
block_batchnorm                MACs=0.000e+00 ACs=3.514e+06 Energy=3.162 uJ
block_residual                 MACs=0.000e+00 ACs=8.784e+05 Energy=0.791 uJ
readout                        MACs=1.478e+08 ACs=3.604e+05 Energy=679.993 uJ
TOTAL                          MACs=2.384e+08 ACs=8.554e+06 Energy=1104.253 uJ

Summary SNN 5M:
Params: 1,095,635
Storage: 4.18 MB
Mean MACs: 238.3813M
Mean ACs: 8.5545M
Mean energy: 1104.2532 uJ
Binning latency: 100 ms


## TCN

In [23]:
def raw_to_effective_length(raw_length, patch_size, patch_stride):
    raw_length = int(raw_length)
    if patch_size <= 0:
        return raw_length
    if raw_length < patch_size:
        return 0
    return ((raw_length - patch_size) // patch_stride) + 1


def compute_tcn_metrics_from_csv(
    raw_lengths_csv,
    input_features,
    output_size,
    hidden_size,
    n_layers,
    patch_size,
    patch_stride,
    tcn_kernel_size,
    tcn_dilation_base=2,
    include_bias=True,
    count_conv_bias_as_ac=True,
    count_input_output_bias_as_ac=True,
    count_residual_adds_as_ac=True,
):
    df = pd.read_csv(raw_lengths_csv)

    D = int(input_features)
    K = int(output_size)
    h = int(hidden_size)
    L = int(n_layers)
    k = int(tcn_kernel_size)
    dilation_base = int(tcn_dilation_base)

    if patch_size > 0:
        d = D * int(patch_size)
    else:
        d = D

    # -------------------------
    # Parameter count
    # -------------------------
    # input_projection = Linear(d -> h)
    # params = d*h + h

    input_params = d * h + (h if include_bias else 0)

    # each TCN block has Conv1d(h -> h, kernel_size=k)
    # params = h*h*k + h
    # dilation does NOT change parameter count.

    conv_params_per_block = h * h * k + (h if include_bias else 0)
    tcn_block_params = L * conv_params_per_block

    # output layer = Linear(h -> K)
    # params = h*K + K

    output_params = h * K + (K if include_bias else 0)

    n_params = input_params + tcn_block_params + output_params

    memory_bits = n_params * BITS_PER_PARAM
    memory_bytes = memory_bits / 8
    memory_kb = memory_bytes / 1024
    memory_mb = memory_bytes / 1024**2

    # -------------------------
    # Per-step operation count
    # -------------------------
    # MACs:
    # input projection: d*h
    # each causal Conv1d block: h*h*k per output timestep
    # output layer: h*K
    #
    # Dilation changes which timesteps are read, but the number of multiplications
    # per output timestep remains h*h*k.

    input_macs_per_step = d * h
    conv_macs_per_block_per_step = h * h * k
    tcn_macs_per_step = L * conv_macs_per_block_per_step
    output_macs_per_step = h * K

    macs_per_step = (
        input_macs_per_step
        + tcn_macs_per_step
        + output_macs_per_step
    )

    # ACs:
    # We count cheap additions separately:
    # - input/output bias additions
    # - conv bias additions
    # - residual additions in each TCN block
    #
    # GELU/dropout/padding/transpose are not counted here, same style as
    # simple MAC/AC model-counting for GRU/LSTM/MLP.

    input_bias_acs_per_step = h if (include_bias and count_input_output_bias_as_ac) else 0
    conv_bias_acs_per_block_per_step = h if (include_bias and count_conv_bias_as_ac) else 0
    residual_acs_per_block_per_step = h if count_residual_adds_as_ac else 0
    output_bias_acs_per_step = K if (include_bias and count_input_output_bias_as_ac) else 0

    acs_per_step = (
        input_bias_acs_per_step
        + L * conv_bias_acs_per_block_per_step
        + L * residual_acs_per_block_per_step
        + output_bias_acs_per_step
    )

    energy_per_step_J = (
        macs_per_step * ENERGY_MAC
        + acs_per_step * ENERGY_AC
    )
    energy_per_step_uJ = energy_per_step_J * 1e6

    # -------------------------
    # Effective sequence length
    # -------------------------

    df["effective_length"] = df["raw_length"].apply(
        lambda x: raw_to_effective_length(x, patch_size, patch_stride)
    )

    df["macs"] = df["effective_length"] * macs_per_step
    df["acs"] = df["effective_length"] * acs_per_step
    df["energy_J"] = df["macs"] * ENERGY_MAC + df["acs"] * ENERGY_AC
    df["energy_mJ"] = df["energy_J"] * 1e3
    df["energy_uJ"] = df["energy_J"] * 1e6

    if patch_size > 0:
        binning_latency_ms = (patch_size - 1) * BIN_SIZE_MS
    else:
        binning_latency_ms = 0

    # -------------------------
    # Receptive field
    # -------------------------
    # Causal TCN receptive field in effective timesteps:
    # RF = 1 + sum_l (kernel_size - 1) * dilation_l
    # dilation_l = dilation_base ** layer_index

    dilations = [dilation_base ** i for i in range(L)]
    receptive_field_effective_steps = 1 + sum((k - 1) * dil for dil in dilations)

    if patch_size > 0:
        receptive_field_raw_bins = (
            patch_size
            + (receptive_field_effective_steps - 1) * patch_stride
        )
    else:
        receptive_field_raw_bins = receptive_field_effective_steps

    receptive_field_ms = receptive_field_raw_bins * BIN_SIZE_MS

    aggregate = {
        "mean_effective_length": df["effective_length"].mean(),
        "median_effective_length": df["effective_length"].median(),
        "p95_effective_length": df["effective_length"].quantile(0.95),

        "mean_macs": df["macs"].mean(),
        "median_macs": df["macs"].median(),
        "p95_macs": df["macs"].quantile(0.95),

        "mean_acs": df["acs"].mean(),
        "median_acs": df["acs"].median(),
        "p95_acs": df["acs"].quantile(0.95),

        "mean_energy_J": df["energy_J"].mean(),
        "median_energy_J": df["energy_J"].median(),
        "p95_energy_J": df["energy_J"].quantile(0.95),

        "mean_energy_uJ": df["energy_uJ"].mean(),
        "median_energy_uJ": df["energy_uJ"].median(),
        "p95_energy_uJ": df["energy_uJ"].quantile(0.95),
    }

    return {
        "D": D,
        "d": d,
        "h": h,
        "L": L,
        "K": K,
        "kernel_size": k,
        "dilation_base": dilation_base,
        "dilations": dilations,

        "input_params": input_params,
        "conv_params_per_block": conv_params_per_block,
        "tcn_block_params": tcn_block_params,
        "output_params": output_params,
        "n_params": n_params,

        "memory_bits": memory_bits,
        "memory_bytes": memory_bytes,
        "memory_kB": memory_kb,
        "memory_MB": memory_mb,

        "input_macs_per_step": input_macs_per_step,
        "conv_macs_per_block_per_step": conv_macs_per_block_per_step,
        "tcn_macs_per_step": tcn_macs_per_step,
        "output_macs_per_step": output_macs_per_step,
        "macs_per_step": macs_per_step,

        "input_bias_acs_per_step": input_bias_acs_per_step,
        "conv_bias_acs_per_block_per_step": conv_bias_acs_per_block_per_step,
        "residual_acs_per_block_per_step": residual_acs_per_block_per_step,
        "output_bias_acs_per_step": output_bias_acs_per_step,
        "acs_per_step": acs_per_step,

        "energy_per_step_J": energy_per_step_J,
        "energy_per_step_uJ": energy_per_step_uJ,

        "patch_size": patch_size,
        "patch_stride": patch_stride,
        "binning_latency_ms": binning_latency_ms,

        "receptive_field_effective_steps": receptive_field_effective_steps,
        "receptive_field_raw_bins": receptive_field_raw_bins,
        "receptive_field_ms": receptive_field_ms,

        "per_utterance_df": df,
        "aggregate": aggregate,
    }


In [24]:
# ===== TCN CONFIG 0.5M =====

patch_size = 10
hidden_size = 64
n_layers = 4
tcn_kernel_size = 7
tcn_dilation_base = 1

tcn_metrics = compute_tcn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=hidden_size,
    n_layers=n_layers,
    patch_size=patch_size,
    patch_stride=patch_stride,
    tcn_kernel_size=tcn_kernel_size,
    tcn_dilation_base=tcn_dilation_base,
    include_bias=True,
    count_conv_bias_as_ac=True,
    count_input_output_bias_as_ac=True,
    count_residual_adds_as_ac=True,
)

agg = tcn_metrics["aggregate"]

print("\n===== TCN 0.5M =====")
print("D:", tcn_metrics["D"])
print("d:", tcn_metrics["d"])
print("h:", tcn_metrics["h"])
print("L:", tcn_metrics["L"])
print("K:", tcn_metrics["K"])
print("Kernel size:", tcn_metrics["kernel_size"])
print("Dilation base:", tcn_metrics["dilation_base"])
print("Dilations:", tcn_metrics["dilations"])

print("\n--- Parameter breakdown ---")
print("Input projection params:", tcn_metrics["input_params"])
print("Conv params per TCN block:", tcn_metrics["conv_params_per_block"])
print("Total TCN block params:", tcn_metrics["tcn_block_params"])
print("Output params:", tcn_metrics["output_params"])

print("\n--- Model size ---")
print("Parameters:", tcn_metrics["n_params"])
print("Parameter count matches Optuna reported count: 10772009")
print("Memory footprint (bits):", tcn_metrics["memory_bits"])
print("Memory footprint (bytes):", tcn_metrics["memory_bytes"])
print("Memory footprint (kB):", tcn_metrics["memory_kB"])
print("Memory footprint (MB):", tcn_metrics["memory_MB"])

print("\n--- Per-step operation breakdown ---")
print("Input projection MACs per timestep:", tcn_metrics["input_macs_per_step"])
print("Conv MACs per block per timestep:", tcn_metrics["conv_macs_per_block_per_step"])
print("Total TCN conv MACs per timestep:", tcn_metrics["tcn_macs_per_step"])
print("Output MACs per timestep:", tcn_metrics["output_macs_per_step"])
print("Total MACs per timestep:", tcn_metrics["macs_per_step"])

print("\n--- Per-step AC breakdown ---")
print("Input bias ACs per timestep:", tcn_metrics["input_bias_acs_per_step"])
print("Conv bias ACs per block per timestep:", tcn_metrics["conv_bias_acs_per_block_per_step"])
print("Residual ACs per block per timestep:", tcn_metrics["residual_acs_per_block_per_step"])
print("Output bias ACs per timestep:", tcn_metrics["output_bias_acs_per_step"])
print("Total ACs per timestep:", tcn_metrics["acs_per_step"])

print("\n--- Per-step energy ---")
print("Energy per timestep (J):", tcn_metrics["energy_per_step_J"])
print("Energy per timestep (uJ):", tcn_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", tcn_metrics["binning_latency_ms"])

print("\n--- TCN receptive field ---")
print("Receptive field effective timesteps:", tcn_metrics["receptive_field_effective_steps"])
print("Receptive field raw bins:", tcn_metrics["receptive_field_raw_bins"])
print("Receptive field ms:", tcn_metrics["receptive_field_ms"])


===== TCN 0.5M =====
D: 512
d: 5120
h: 64
L: 4
K: 41
Kernel size: 7
Dilation base: 1
Dilations: [1, 1, 1, 1]

--- Parameter breakdown ---
Input projection params: 327744
Conv params per TCN block: 28736
Total TCN block params: 114944
Output params: 2665

--- Model size ---
Parameters: 445353
Parameter count matches Optuna reported count: 10772009
Memory footprint (bits): 14251296
Memory footprint (bytes): 1781412.0
Memory footprint (kB): 1739.66015625
Memory footprint (MB): 1.6988868713378906

--- Per-step operation breakdown ---
Input projection MACs per timestep: 327680
Conv MACs per block per timestep: 28672
Total TCN conv MACs per timestep: 114688
Output MACs per timestep: 2624
Total MACs per timestep: 444992

--- Per-step AC breakdown ---
Input bias ACs per timestep: 64
Conv bias ACs per block per timestep: 64
Residual ACs per block per timestep: 64
Output bias ACs per timestep: 41
Total ACs per timestep: 617

--- Per-step energy ---
Energy per timestep (J): 2.0475185e-06
Energy 

In [25]:
# ===== TCN CONFIG 2M =====

patch_size = 10
hidden_size = 256
n_layers = 5
tcn_kernel_size = 3
tcn_dilation_base = 1

tcn_metrics = compute_tcn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=hidden_size,
    n_layers=n_layers,
    patch_size=patch_size,
    patch_stride=patch_stride,
    tcn_kernel_size=tcn_kernel_size,
    tcn_dilation_base=tcn_dilation_base,
    include_bias=True,
    count_conv_bias_as_ac=True,
    count_input_output_bias_as_ac=True,
    count_residual_adds_as_ac=True,
)

agg = tcn_metrics["aggregate"]

print("\n===== TCN 2M =====")
print("D:", tcn_metrics["D"])
print("d:", tcn_metrics["d"])
print("h:", tcn_metrics["h"])
print("L:", tcn_metrics["L"])
print("K:", tcn_metrics["K"])
print("Kernel size:", tcn_metrics["kernel_size"])
print("Dilation base:", tcn_metrics["dilation_base"])
print("Dilations:", tcn_metrics["dilations"])

print("\n--- Parameter breakdown ---")
print("Input projection params:", tcn_metrics["input_params"])
print("Conv params per TCN block:", tcn_metrics["conv_params_per_block"])
print("Total TCN block params:", tcn_metrics["tcn_block_params"])
print("Output params:", tcn_metrics["output_params"])

print("\n--- Model size ---")
print("Parameters:", tcn_metrics["n_params"])
print("Parameter count matches Optuna reported count: 5265449")
print("Memory footprint (bits):", tcn_metrics["memory_bits"])
print("Memory footprint (bytes):", tcn_metrics["memory_bytes"])
print("Memory footprint (kB):", tcn_metrics["memory_kB"])
print("Memory footprint (MB):", tcn_metrics["memory_MB"])

print("\n--- Per-step operation breakdown ---")
print("Input projection MACs per timestep:", tcn_metrics["input_macs_per_step"])
print("Conv MACs per block per timestep:", tcn_metrics["conv_macs_per_block_per_step"])
print("Total TCN conv MACs per timestep:", tcn_metrics["tcn_macs_per_step"])
print("Output MACs per timestep:", tcn_metrics["output_macs_per_step"])
print("Total MACs per timestep:", tcn_metrics["macs_per_step"])

print("\n--- Per-step AC breakdown ---")
print("Input bias ACs per timestep:", tcn_metrics["input_bias_acs_per_step"])
print("Conv bias ACs per block per timestep:", tcn_metrics["conv_bias_acs_per_block_per_step"])
print("Residual ACs per block per timestep:", tcn_metrics["residual_acs_per_block_per_step"])
print("Output bias ACs per timestep:", tcn_metrics["output_bias_acs_per_step"])
print("Total ACs per timestep:", tcn_metrics["acs_per_step"])

print("\n--- Per-step energy ---")
print("Energy per timestep (J):", tcn_metrics["energy_per_step_J"])
print("Energy per timestep (uJ):", tcn_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", tcn_metrics["binning_latency_ms"])

print("\n--- TCN receptive field ---")
print("Receptive field effective timesteps:", tcn_metrics["receptive_field_effective_steps"])
print("Receptive field raw bins:", tcn_metrics["receptive_field_raw_bins"])
print("Receptive field ms:", tcn_metrics["receptive_field_ms"])


===== TCN 2M =====
D: 512
d: 5120
h: 256
L: 5
K: 41
Kernel size: 3
Dilation base: 1
Dilations: [1, 1, 1, 1, 1]

--- Parameter breakdown ---
Input projection params: 1310976
Conv params per TCN block: 196864
Total TCN block params: 984320
Output params: 10537

--- Model size ---
Parameters: 2305833
Parameter count matches Optuna reported count: 5265449
Memory footprint (bits): 73786656
Memory footprint (bytes): 9223332.0
Memory footprint (kB): 9007.16015625
Memory footprint (MB): 8.79605484008789

--- Per-step operation breakdown ---
Input projection MACs per timestep: 1310720
Conv MACs per block per timestep: 196608
Total TCN conv MACs per timestep: 983040
Output MACs per timestep: 10496
Total MACs per timestep: 2304256

--- Per-step AC breakdown ---
Input bias ACs per timestep: 256
Conv bias ACs per block per timestep: 256
Residual ACs per block per timestep: 256
Output bias ACs per timestep: 41
Total ACs per timestep: 2857

--- Per-step energy ---
Energy per timestep (J): 1.06021488

In [26]:
# ===== TCN CONFIG 5M =====

patch_size = 6
hidden_size = 512
n_layers = 2
tcn_kernel_size = 7
tcn_dilation_base = 1


tcn_metrics = compute_tcn_metrics_from_csv(
    raw_lengths_csv=RAW_LENGTHS_CSV,
    input_features=INPUT_FEATURES,
    output_size=OUTPUT_SIZE,
    hidden_size=hidden_size,
    n_layers=n_layers,
    patch_size=patch_size,
    patch_stride=patch_stride,
    tcn_kernel_size=tcn_kernel_size,
    tcn_dilation_base=tcn_dilation_base,
    include_bias=True,
    count_conv_bias_as_ac=True,
    count_input_output_bias_as_ac=True,
    count_residual_adds_as_ac=True,
)

agg = tcn_metrics["aggregate"]

print("\n===== TCN 5M =====")
print("D:", tcn_metrics["D"])
print("d:", tcn_metrics["d"])
print("h:", tcn_metrics["h"])
print("L:", tcn_metrics["L"])
print("K:", tcn_metrics["K"])
print("Kernel size:", tcn_metrics["kernel_size"])
print("Dilation base:", tcn_metrics["dilation_base"])
print("Dilations:", tcn_metrics["dilations"])

print("\n--- Parameter breakdown ---")
print("Input projection params:", tcn_metrics["input_params"])
print("Conv params per TCN block:", tcn_metrics["conv_params_per_block"])
print("Total TCN block params:", tcn_metrics["tcn_block_params"])
print("Output params:", tcn_metrics["output_params"])

print("\n--- Model size ---")
print("Parameters:", tcn_metrics["n_params"])
print("Parameter count matches Optuna reported count: 2305833")
print("Memory footprint (bits):", tcn_metrics["memory_bits"])
print("Memory footprint (bytes):", tcn_metrics["memory_bytes"])
print("Memory footprint (kB):", tcn_metrics["memory_kB"])
print("Memory footprint (MB):", tcn_metrics["memory_MB"])

print("\n--- Per-step operation breakdown ---")
print("Input projection MACs per timestep:", tcn_metrics["input_macs_per_step"])
print("Conv MACs per block per timestep:", tcn_metrics["conv_macs_per_block_per_step"])
print("Total TCN conv MACs per timestep:", tcn_metrics["tcn_macs_per_step"])
print("Output MACs per timestep:", tcn_metrics["output_macs_per_step"])
print("Total MACs per timestep:", tcn_metrics["macs_per_step"])

print("\n--- Per-step AC breakdown ---")
print("Input bias ACs per timestep:", tcn_metrics["input_bias_acs_per_step"])
print("Conv bias ACs per block per timestep:", tcn_metrics["conv_bias_acs_per_block_per_step"])
print("Residual ACs per block per timestep:", tcn_metrics["residual_acs_per_block_per_step"])
print("Output bias ACs per timestep:", tcn_metrics["output_bias_acs_per_step"])
print("Total ACs per timestep:", tcn_metrics["acs_per_step"])

print("\n--- Per-step energy ---")
print("Energy per timestep (J):", tcn_metrics["energy_per_step_J"])
print("Energy per timestep (uJ):", tcn_metrics["energy_per_step_uJ"])

print("\n--- Effective sequence length ---")
print("Mean effective length:", agg["mean_effective_length"])
print("Median effective length:", agg["median_effective_length"])
print("P95 effective length:", agg["p95_effective_length"])

print("\n--- Operations per inference ---")
print("Mean MACs:", agg["mean_macs"])
print("Median MACs:", agg["median_macs"])
print("P95 MACs:", agg["p95_macs"])
print("Mean ACs:", agg["mean_acs"])
print("Median ACs:", agg["median_acs"])
print("P95 ACs:", agg["p95_acs"])

print("\n--- Energy per inference ---")
print("Mean energy (J):", agg["mean_energy_J"])
print("Mean energy (uJ):", agg["mean_energy_uJ"])
print("Median energy (uJ):", agg["median_energy_uJ"])
print("P95 energy (uJ):", agg["p95_energy_uJ"])

print("\n--- Binning latency ---")
print("Binning latency (ms):", tcn_metrics["binning_latency_ms"])

print("\n--- TCN receptive field ---")
print("Receptive field effective timesteps:", tcn_metrics["receptive_field_effective_steps"])
print("Receptive field raw bins:", tcn_metrics["receptive_field_raw_bins"])
print("Receptive field ms:", tcn_metrics["receptive_field_ms"])


===== TCN 5M =====
D: 512
d: 3072
h: 512
L: 2
K: 41
Kernel size: 7
Dilation base: 1
Dilations: [1, 1]

--- Parameter breakdown ---
Input projection params: 1573376
Conv params per TCN block: 1835520
Total TCN block params: 3671040
Output params: 21033

--- Model size ---
Parameters: 5265449
Parameter count matches Optuna reported count: 2305833
Memory footprint (bits): 168494368
Memory footprint (bytes): 21061796.0
Memory footprint (kB): 20568.16015625
Memory footprint (MB): 20.08609390258789

--- Per-step operation breakdown ---
Input projection MACs per timestep: 1572864
Conv MACs per block per timestep: 1835008
Total TCN conv MACs per timestep: 3670016
Output MACs per timestep: 20992
Total MACs per timestep: 5263872

--- Per-step AC breakdown ---
Input bias ACs per timestep: 512
Conv bias ACs per block per timestep: 512
Residual ACs per block per timestep: 512
Output bias ACs per timestep: 41
Total ACs per timestep: 2601

--- Per-step energy ---
Energy per timestep (J): 2.421615209